# Inflation Forecasting — AR-OLS + Random Walk

Simple, interpretable baselines for 1-month-ahead PCEPI monthly log-diff × 100:

- **AR-OLS** — OLS on 1–12 lags of the PCEPI series (univariate).
- **Naive random walk** — predict previous month's value.

Test split matches the other notebooks (`test_start='2025-06-01'`).
Predictions are saved to `results/` for `model_comparison.ipynb`.

In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from fred_md_utils import (
    configure_plots, default_paths, get_splits,
    load_fred_md_raw,
    TEST_START, VAL_START,
)

warnings.filterwarnings('ignore')
configure_plots()
VINTAGE_DIR, RESULTS_DIR = default_paths()

AR_LAGS = 12


## Data Loading

In [ ]:
# Shared split (matches other model notebooks exactly)
vintage_file, _, _, _, _, X_test, y_test, _ = get_splits(
    VINTAGE_DIR, horizon=1, n_lags=0,
)

# Univariate PCEPI log-diff x100 series (matches build_dataset_from_csv's target)
df_raw, _ = load_fred_md_raw(vintage_file)
pcepi_series = (np.log(df_raw['PCEPI']).diff() * 100).dropna()
pcepi_series.name = 'PCEPI_logdiff_x100'

test_dates     = y_test.index
train_only_end = pd.Timestamp(VAL_START) - pd.DateOffset(months=1)  # coefficients fit up to here

print(f'PCEPI series : {pcepi_series.index.min().date()} -> {pcepi_series.index.max().date()}  ({len(pcepi_series)} obs)')
print(f'Train ends   : {train_only_end.date()}  (coefficients frozen here)')
print(f'Val window   : {VAL_START} -> {(test_dates.min() - pd.DateOffset(months=1)).date()}')
print(f'Test window  : {test_dates.min().date()} -> {test_dates.max().date()}   n_test = {len(test_dates)}')


## Model 1 — AR-OLS (PCEPI lags only)

Plain OLS with 12 lags of the PCEPI log-diff series.

In [ ]:
def build_lag_matrix(series, lags):
    df = pd.DataFrame({f'lag{i}': series.shift(i) for i in range(1, lags + 1)})
    df['y'] = series
    return df.dropna()

ar_df = build_lag_matrix(pcepi_series, AR_LAGS)

# ── Fit coefficients on TRAIN only (data before VAL_START) ───────────────────
ar_train = ar_df.loc[:train_only_end]

X_ar = ar_train.drop(columns='y').values
y_ar = ar_train['y'].values
ar_model = LinearRegression(fit_intercept=True)
ar_model.fit(X_ar, y_ar)
print(f'Intercept : {ar_model.intercept_:.6f}')
coef_df = pd.DataFrame({'feature': ar_train.drop(columns='y').columns, 'coef': ar_model.coef_})
print(coef_df.to_string(index=False))
print(f'\nCoefficients fit on training data: {ar_train.index.min().date()} -> {ar_train.index.max().date()}  (n={len(ar_train)})')

# ── In-sample (train) R² ─────────────────────────────────────────────────────
train_r2 = ar_model.score(X_ar, y_ar)
print(f'\nAR-OLS({AR_LAGS}) train R²  : {train_r2:.4f}')

# ── Validation R² (VAL_START up to test start) ───────────────────────────────
val_end_date   = test_dates.min() - pd.DateOffset(months=1)
val_idx        = ar_df.loc[pd.Timestamp(VAL_START):val_end_date].index
val_rows       = ar_df.loc[val_idx]
X_val_ar       = val_rows.drop(columns='y').values
y_val_true     = val_rows['y'].values
y_val_hat      = ar_model.predict(X_val_ar)
val_r2         = r2_score(y_val_true, y_val_hat)
val_rmse       = np.sqrt(mean_squared_error(y_val_true, y_val_hat))
print(f'AR-OLS({AR_LAGS}) val   R²  : {val_r2:.4f}   RMSE = {val_rmse:.4f}  '
      f'({val_idx.min().date()} -> {val_idx.max().date()},  n={len(val_idx)})')

# ── Test predictions (same frozen coefficients) ───────────────────────────────
X_test_ar = ar_df.loc[test_dates].drop(columns='y').values
ar_preds  = ar_model.predict(X_test_ar)

ar_rmse  = np.sqrt(mean_squared_error(y_test.values, ar_preds))
ar_r2    = r2_score(y_test.values, ar_preds)
print(f'AR-OLS({AR_LAGS}) test  R²  : {ar_r2:.4f}   RMSE = {ar_rmse:.4f}  '
      f'({test_dates.min().date()} -> {test_dates.max().date()},  n={len(test_dates)})')

if train_r2 > 0.05 and val_r2 < 0:
    print('\n⚠  Train R² positive, val/test R² negative — classic out-of-sample '
          'overfitting.  The AR coefficients fit historical patterns but '
          'do not generalise well to the post-COVID inflation regime.')

## Model 2 — Naive random walk

In [ ]:
naive_preds = []
for date in test_dates:
    prev_date = date - pd.DateOffset(months=1)
    if prev_date in pcepi_series.index:
        naive_preds.append(float(pcepi_series.loc[prev_date]))
    else:
        naive_preds.append(float(pcepi_series.iloc[-1]))

naive_preds = np.array(naive_preds)
naive_rmse = np.sqrt(mean_squared_error(y_test.values, naive_preds))
print(f'Naive random walk test RMSE: {naive_rmse:.4f}')

## Metrics Table + Overlay Plot

In [ ]:
def metrics(name, y_true, y_pred):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    return {
        'Model':    name,
        'RMSE':     float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'MAE':      float(mean_absolute_error(y_true, y_pred)),
        'R2':       float(r2_score(y_true, y_pred)),
        'MAPE (%)': float(np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100),
    }

results_df = pd.DataFrame([
    metrics(f'AR-OLS({AR_LAGS})', y_test.values, ar_preds),
    metrics('Naive (random walk)', y_test.values, naive_preds),
]).sort_values('RMSE').reset_index(drop=True)

print(f'Test period: {test_dates.min().date()} -> {test_dates.max().date()}   n = {len(test_dates)}')
print('Metrics scale: monthly log-diff x 100 (pp of monthly PCE inflation)')
print()
print(results_df.to_string(index=False, float_format='{:.4f}'.format))

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(test_dates, y_test.values, 'o-', color='black', label='Actual', linewidth=2, markersize=5)
ax.plot(test_dates, ar_preds,    's--', color='darkorange', label=f'AR-OLS({AR_LAGS})', markersize=5)
ax.plot(test_dates, naive_preds, 's--', color='seagreen',   label='Naive RW', markersize=5)
ax.axhline(0, color='grey', linewidth=0.5)
ax.set_title('AR-OLS + Naive RW vs Actual PCEPI Monthly Log-Diff x 100')
ax.set_ylabel('pp / month')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
fig.tight_layout()
plt.show()

## Save Predictions for `model_comparison.ipynb`

In [ ]:
dates_arr = np.array(test_dates, dtype='datetime64[ns]')

np.save(os.path.join(RESULTS_DIR, 'ar_ols_preds.npy'), ar_preds)
np.save(os.path.join(RESULTS_DIR, 'ar_ols_dates.npy'), dates_arr)

np.save(os.path.join(RESULTS_DIR, 'naive_preds.npy'),  naive_preds)
np.save(os.path.join(RESULTS_DIR, 'naive_dates.npy'),  dates_arr)

# Clean up stale MV-OLS artifacts from earlier runs
for k in ['mv_ols_preds', 'mv_ols_dates']:
    p = os.path.join(RESULTS_DIR, f'{k}.npy')
    if os.path.exists(p):
        os.remove(p)
        print(f'Removed stale {p}')

print('Saved to results/:')
for k in ['ar_ols_preds', 'ar_ols_dates', 'naive_preds', 'naive_dates']:
    p = os.path.join(RESULTS_DIR, f'{k}.npy')
    print(f'  {p}   shape = {np.load(p, allow_pickle=False).shape}')